# code

> fill kosha from a repo, grep the working tree, and federate all of it with the prose

In [ ]:
#| default_exp code

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

[kosha](https://github.com/vedicreader/kosha) is the code half of the vault: AST chunks, symbol names, call graph with PageRank. `index_code` fills it; `symbol`, `where_to_add`, and `grep` (via [rgapi](https://github.com/AnswerDotAI/rgapi)) read it back.


In [ ]:
#| export
from fastcore.all import AttrDict, L, Path, patch
from litesearch import rrf_all
from vishalakshi.core import Vault

In [ ]:
#| export
@patch
def kosha(self:Vault,
          dir:str=None,              # repo root; defaults to the cwd repo
          share_encoder:bool=False,  # embed code with the vault's encoder instead of kosha's
          **kw                       # forwarded to Kosha()
):
    'The `Kosha` for `dir`, cached on the vault so `index_code` and `code_search` hit one store.'
    from kosha import Kosha
    k, root = getattr(self, '_k', None), Path(dir).resolve() if dir else None
    if k is None or (root and k.root != root):
        # `Index` holds the encoder; kosha wants a callable that returns one
        if share_encoder and self.encoder: kw['efn'] = lambda: self.encoder
        self._k = k = Kosha(dir, **kw)
    return k

@patch
def index_code(self:Vault,
               dir:str=None,      # repo to index
               graph:bool=True,   # also build the AST call graph (callers, callees, PageRank)
               env:bool=False,    # also index installed packages (slow the first time)
               force:bool=False,  # reindex everything, graph included, rather than what changed
               verbose:bool=False,
               **kw               # forwarded to kosha's `sync`
) -> dict:
    "Point the vault at a repo and fill kosha's stores from it."
    k = self.kosha(dir)
    k.sync(dir=dir, repo=True, env=env, graph=graph, force=force, verbose=verbose, **kw)
    return k.status()

`grep` is the leg neither of the others can serve. Embeddings generalise and FTS5 stems, so an
identifier appearing verbatim in a file the vault never ingested, or ingested an older copy of, is
invisible to both. It reads the working tree as it is right now.

In [ ]:
#| export
@patch
def code_search(self:Vault, q:str, limit:int=10, dir:str=None, **kw) -> L:
    "Search code through kosha: FTS + ANN, fused and rank-boosted. Takes kosha's `key:value` filters."
    return self.kosha(dir).context(q, limit=limit, **kw)

@patch
def symbol(self:Vault, name:str, depth:int=1, dir:str=None) -> AttrDict:
    'A symbol in the call graph: its file, PageRank and degree, plus its callers and callees.'
    k = self.kosha(dir)
    return AttrDict(node=name, info=dict(k.ni(name) or {}), neighbors=L(k.neighbors(name, depth)))

@patch
def where_to_add(self:Vault, description:str, limit:int=5, dir:str=None) -> L:
    'Where in the indexed repo a described change belongs: kosha ranking over the call graph.'
    return self.kosha(dir).where_to_add(description, limit=limit)

@patch
def grep(self:Vault,
         pattern:str,       # a ripgrep regex
         dir:str='.',       # tree to search
         limit:int=20,      # matching lines returned
         **kw               # forwarded to rgapi.rg (glob=, ext=, context=, hidden=, ...)
) -> L:
    'Exact matches in the files on disk, through ripgrep. `.gitignore` applies.'
    from rgapi import rg
    return _rg(rg(pattern, root=dir, max_results=limit, smart_case=True, **kw))

### Fusing legs that share no vector space

The vault embeds prose, kosha embeds identifiers, ripgrep embeds nothing. Legs share no vector space, so `federate` fuses rankings with RRF and never distances.


In [ ]:
#| export
def _prose(v, q, n, kind=None, source:str='prose') -> L:
    'Vault sections, normalised to the federated row shape. `source` names the shelf they came from.'
    return L(v.sections(q, limit=n, kind=kind)).map(
        lambda s: AttrDict(source=source, ref=s['node_id'], title=s['title'], where=s['breadcrumb'],
                           text=' '.join(s['snippets'])[:600], open=f"read({s['node_id']!r})"))

def _code(rows, leg:str) -> L:
    'kosha hits, normalised to the federated row shape.'
    def row(r):
        m = dict(r.get('metadata') or {})
        mod, path, ln = m.get('mod_name', ''), m.get('path', ''), m.get('lineno')
        return AttrDict(source=leg, ref=mod or path, title=mod or (Path(path).name if path else ''),
                        where=f'{path}:{ln}' if path and ln else (path or mod),
                        text=(r.get('content') or '')[:600],
                        open=f'symbol({mod!r})' if mod else f'open {path}')
    return L(rows).map(row)

def _rg(matches) -> L:
    'ripgrep lines, normalised to the federated row shape.'
    return L(matches).map(lambda m: AttrDict(source='grep', ref=f'{m.path}:{m.line_number}',
                                             title=Path(m.path).name, where=f'{m.path}:{m.line_number}',
                                             text=m.line.strip()[:600], open=f'open {m.path}'))

@patch
def federate(self:Vault,
             q:str,             # the query
             limit:int=12,      # fused hits returned
             prose:bool=True,   # the vault's own documents, papers, notes
             repo:bool=True,    # kosha's repo index
             env:bool=False,    # kosha's installed-package index
             grep:bool=True,    # ripgrep over the working tree
             kind:str=None,     # restrict the prose leg to some KINDS
             shelves=True,      # other shelves as their own legs: True for all, a list to pick, None for none
             weights:dict=None, # per-leg RRF weights, e.g. {'prose':1.0,'repo':1.5}
             dir:str=None,      # repo for the code and grep legs
             per_leg:int=None,  # hits pulled from each leg before fusion
) -> AttrDict:
    'One ranked answer across prose, indexed code and the files on disk, fused by RRF.'
    n, legs = per_leg or max(limit, 10), {}
    def leg(nm, f):
        try: legs[nm] = f()
        except Exception as e: legs[f'{nm}_error'] = f'{type(e).__name__}: {str(e)[:120]}'
    if prose: leg('prose', lambda: _prose(self, q, n, kind=kind))
    names = (self.shelves().filter(lambda s: s['store'] != self.name and s['docs']).attrgot('store')
             if shelves is True else L(shelves))
    for sh in names:
        s = sh if isinstance(sh, Vault) else self.shelf(sh)
        leg(nm := f'shelf:{s.name}', lambda s=s, nm=nm: _prose(s, q, n, kind=kind, source=nm))
    if repo:  leg('repo', lambda: _code(self.kosha(dir).repo_context(q, limit=n), 'repo'))
    if env:   leg('env', lambda: _code(self.kosha(dir).env_context(q, limit=n), 'env'))
    if grep:  leg('grep', lambda: self.grep(q, dir or '.', limit=n))
    lists = {nm: rows for nm, rows in legs.items() if isinstance(rows, L) and rows}
    for nm, rows in lists.items():
        for i, r in enumerate(rows): r['_fid'] = f'{nm}:{r.ref or i}'
    fused = rrf_all(list(lists.values()), limit=limit, id_key='_fid',
                    weights=[(weights or {}).get(nm, 1.0) for nm in lists])
    return AttrDict(query=q, hits=L(fused).map(AttrDict),
                    legs={nm: (len(r) if isinstance(r, L) else r) for nm, r in legs.items()},
                    note=f"RRF over {', '.join(lists) or 'nothing'}; the legs use different "
                         f"encoders, so ranks are fused, not distances")

### Code in an answer

`context(..., code=n)` appends code sections beside prose when `.kosha/code.db` is on disk. Code citations use `path:line`, not a `node_id`.


In [ ]:
#| export
def kosha_indexed(dir:str=None) -> bool:
    'Has kosha already indexed this repo? A file check, so asking costs nothing.'
    from litesearch import repo_root
    d = Path(dir) if dir else repo_root()
    return bool(d) and (Path(d)/'.kosha'/'code.db').exists()

def code_sections(v,                 # the Vault
                  q:str,             # the question
                  n:int=4,           # code hits to keep
                  dir:str=None,      # repo for the kosha and ripgrep legs
                  **kw               # forwarded to federate
) -> L:
    'Federated code hits, shaped like vault sections so the rest of `ask` cannot tell the difference.'
    if not getattr(v, 'federate', None): return L()
    f = v.federate(q, limit=n, prose=False, dir=dir, **kw)
    return L(f.hits).map(lambda h: AttrDict(
        node_id=None, doc_id=None, title=h.get('title') or h.get('ref'), pages=None,
        breadcrumb=f"{h.get('source','code')} › {h.get('where') or h.get('ref')}",
        filename=h.get('where') or h.get('ref'), text=h.get('text') or '', via=h.get('source')))

## Try it

Each leg is optional and each fails on its own: with no repo indexed, `legs` says so and the
others still answer.

In [ ]:
v = Vault(':memory:')
v.note('litesearch fuses the FTS and vector legs with reciprocal rank fusion.')
f = v.federate('rank fusion', repo=False, grep=False)
f.hits[0].where, f.legs, f.note

/Users/71293/code/personal/orgs/vishalakshi/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


('litesearch fuses the FTS and vector legs with reciprocal rank fusion.',
 {'prose': 1},
 'RRF over prose; the legs use different encoders, so ranks are fused, not distances')

In [ ]:
test_eq(f.legs['prose'], 1)
test_eq(f.hits[0].source, 'prose')
# every leg off: empty, not an error (shelves=None too, or an existing shelf would answer)
test_eq(v.federate('rank fusion', prose=False, repo=False, grep=False, shelves=None).hits, [])
assert v.grep('reciprocal rank fusion', '.')                                     # ripgrep reads the disk, not the vault

In [ ]:
#| hide
# a code hit has to be indistinguishable from a section, or `mk_prompt` and `cited` need special cases
rows = code_sections(v, 'reciprocal rank fusion', n=2, repo=False, dir='.')
assert rows, 'ripgrep alone should find this'
for r in rows:
    test_eq(set(r) >= {'node_id', 'doc_id', 'title', 'pages', 'breadcrumb', 'filename', 'text'}, True)
    test_eq(r.node_id, None)      # the honest signal: read() cannot open a path:line
    assert r.filename and r.text

# asking costs nothing when kosha has not indexed anything: a file check, never a Kosha()
from tempfile import mkdtemp
test_eq(kosha_indexed(mkdtemp()), False)
test_eq(kosha_indexed('.'), (Path('.kosha')/'code.db').exists())

# and `context` splices them in *after* the prose, so the numbering an answer cites cannot shift
c = v.context('reciprocal rank fusion', sections=2, related=0, code=2, dir='.')
test_eq(c.code, len(c.results) - len([r for r in c.results if r.node_id]))
assert c.results[0].node_id, 'prose must come first'
test_eq(v.context('reciprocal rank fusion', related=0, code=0).code, 0)   # 0 is off, not auto

In [ ]:
#| hide
# a shelf joins as its own leg, because it is its own vector space: fused by rank, never by distance
sh = v.shelf('papers', offline=True)
sh.add('Contextual chunk embeddings, evaluated on BEIR.', 'a paper')
f2 = v.federate('chunk embeddings', repo=False, grep=False)   # shelves=True is the default
test_eq(sorted(f2.legs), ['prose', 'shelf:papers'])
assert any(h.source == 'shelf:papers' for h in f2.hits), f2.hits
test_eq(v.federate('chunk embeddings', repo=False, grep=False, shelves=[sh]).legs['shelf:papers'], 1)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()